# Trees syndromes

In [38]:
import pandas as pd
import numpy as np
from pyecharts import options as opts
from pyecharts.charts import Tree, Page, Tab, Grid

In [39]:
data= pd.read_excel("./semio_psy_syndromes_clean.xlsx")
data = data.replace({np.nan: None})
data = data.replace({"\n":"\n"},regex=True)

display(data)

,Niveau 1,Niveau 2,Niveau 3,Niveau 4,Niveau 5,Niveau 6,Syndrome positif,Syndrome de désorganisation,Syndrome négatif,Syndrome dépressif,Syndrome maniaque,Syndrome anxieux,Syndrome suicidaire,Syndrome catatonique,Syndrome d'agitation,Syndrome cognitif,Nombre d'occurrences
0,cognition,perception,hallucination psychosensorielles,hallucinations acoustico verbales,None,None,x,None,None,None,None,None,None,None,None,None,1
1,cognition,perception,hallucination psychosensorielles,hallucinations auditives,None,None,x,None,None,None,None,None,None,None,None,None,1
2,cognition,perception,hallucination psychosensorielles,hallucinations visuelles,None,None,x,None,None,None,None,None,None,None,None,None,1
3,cognition,perception,hallucination psychosensorielles,hallucinations olfactives,None,None,x,None,None,None,None,None,None,None,None,None,1
4,cognition,perception,hallucination psychosensorielles,hallucinations gustatives,None,None,x,None,None,None,None,None,None,None,None,None,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
239,comportement,fonctions physiologiques,alimentation,diminuée,anorexie,None,None,None,None,x,x,None,None,x,None,None,3
240,comportement,fonctions physiologiques,alimentation,augmentée,hyperphagie,None,None,None,None,x,x,None,None,None,None,None,2
241,comportement,fonctions physiologiques,alimentation,altérée,boulimie,None,None,None,None,None,None,None,None,None,None,None,0
242,comportement,fonctions physiologiques,alimentation,altérée,pica,None,None,None,None,None,None,None,None,None,None,None,0


In [40]:
data = data.replace(
    {
        'discours (pensée et langage)':'discours\n(pensée\net langage)',
        'fonctions physiologiques':'fonctions\nphysiologiques',
        'hallucinations intra psychiques':'hallucinations\nintra psychiques', 
        'hallucination psychosensorielles':'hallucination\npsychosensorielles', 
        'phrase ou mot idiosyncrasique':'phrase ou mot\nidiosyncrasique',
        'idées liées à l\'humeur':'idées liées \nà l\'humeur',
        'pragmatique du langage':'pragmatique\ndu langage'
    }
)

# Syndrome profiles

In [41]:
# reformating data
df_res = pd.DataFrame(np.zeros((0,7)), columns=data.columns[0:6].to_list()+['Syndrome'])
for syndrome in data.columns[6:len(data.columns)-1] : 
    print(syndrome)
    df_syndrome = data.loc[data[syndrome] == "x",data.columns[0:6]]
    df_syndrome['Syndrome'] = syndrome
    df_res = pd.concat((df_res, df_syndrome), axis = 0)

Syndrome positif
Syndrome de désorganisation
Syndrome négatif
Syndrome dépressif
Syndrome maniaque
Syndrome anxieux
Syndrome suicidaire
Syndrome catatonique
Syndrome d'agitation
Syndrome cognitif


In [42]:
df_res

,Niveau 1,Niveau 2,Niveau 3,Niveau 4,Niveau 5,Niveau 6,Syndrome
0,cognition,perception,hallucination\npsychosensorielles,hallucinations acoustico verbales,None,None,Syndrome positif
1,cognition,perception,hallucination\npsychosensorielles,hallucinations auditives,None,None,Syndrome positif
2,cognition,perception,hallucination\npsychosensorielles,hallucinations visuelles,None,None,Syndrome positif
3,cognition,perception,hallucination\npsychosensorielles,hallucinations olfactives,None,None,Syndrome positif
4,cognition,perception,hallucination\npsychosensorielles,hallucinations gustatives,None,None,Syndrome positif
...,...,...,...,...,...,...,...
131,cognition,jugement,altéré,capacité d'abstraction altérée,None,None,Syndrome cognitif
132,cognition,cognition sociale,altération de la théorie de l'esprit,None,None,None,Syndrome cognitif
133,cognition,cognition sociale,difficulté dans la reconnaissance des émotions,None,None,None,Syndrome cognitif
134,cognition,cognition sociale,biais d'attribution,None,None,None,Syndrome cognitif


In [43]:

def convertir(d):
    return [{"name": k, "children": convertir(v)} for k, v in d.items()]

COULEURS = {
    "comportement": "##65C5CA",
    "affect": "#f89c74",
    "cognition": "#f7cf71"
}
DEFAUT = "#CCCCCC"
PAS = 0.3   # éclaircissement

def eclaircir(hex_c, f):
    h = hex_c.lstrip("#")
    rgb = [int(h[i:i+2], 16) for i in (0, 2, 4)]
    return "#%02x%02x%02x" % tuple(round(v + (255 - v) * f) for v in rgb)

def colorier(noeud, c, prof=0):
    cc = eclaircir(c, min(PAS * prof, 0.7))
    noeud["itemStyle"] = {"color": cc}
    noeud["lineStyle"] = {"color": cc, "width": 2}
    for e in noeud.get("children", []):
        colorier(e, c, prof + 1)

tab = Tab(page_title="Syndromes (fr)")
for syndrome in np.unique(df_res.Syndrome) : 
    df = df_res[df_res.Syndrome == syndrome]
    df = df.drop("Syndrome",axis = 1)
    arbre_dict = {}
    for chemin in df.itertuples(index=False):
        noeud = arbre_dict
        for nom in chemin:
            if pd.notna(nom):          # niveau absent : on arrête ce chemin
                noeud = noeud.setdefault(nom, {})

    racine = {"name": "", "children": convertir(arbre_dict)}

    for branche in racine["children"]:
        colorier(branche, COULEURS.get(branche["name"], DEFAUT))


    arbre = Tree(init_opts=opts.InitOpts(width="800px", height="800px"))
    arbre.add("", [racine], layout="radial",initial_tree_depth=4,label_opts=opts.LabelOpts(font_size=8))
    arbre.set_global_opts(title_opts=opts.TitleOpts(title=syndrome))
    #page.add(arbre)
    tab.add(arbre, syndrome)
tab.render("../html/syndromes_trees.html")
tab.render_notebook()
